# Golden Confusion Matrix (Task1 + Task2)

Notebook para evaluar **un modelo que tú elijas** sobre golden set y visualizar matriz de confusión bonita.

## Requisitos del modelo
La ruta del modelo debe contener al menos:
- `config.json`
- `tokenizer.json` (o tokenizer equivalente)
- `model.safetensors` (o `pytorch_model.bin`)


In [ ]:
!pip -q install transformers torch scikit-learn pandas matplotlib seaborn

In [ ]:
import re
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, recall_score

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=0.95)

In [ ]:
# ==========================
# CONFIGURA TUS RUTAS AQUÍ
# ==========================
TASK1_GOLD_CSV = '/content/task1_gold_labeled.csv'
TASK2_GOLD_CSV = '/content/task2_gold_labeled.csv'

# Puedes usar el mismo modelo o diferentes
TASK1_MODEL_DIR = '/content/model_task1'
TASK2_MODEL_DIR = '/content/model_task2'

TASK1_BATCH_SIZE = 16
TASK2_BATCH_SIZE = 16
TASK2_THRESHOLD = 0.5

# Etiquetas retóricas esperadas para Task1
TASK1_LABEL_ORDER = ['INTRO', 'BACK', 'METH', 'RES', 'DISC', 'CONTR', 'LIM', 'CONC']

In [ ]:
def clean_text(x):
    return re.sub(r'\s+', ' ', str(x or '')).strip()

def norm_task1_label(x):
    y = str(x or '').strip().upper()
    if y == 'RESU':
        return 'RES'
    return y

def load_model(model_dir):
    model_dir = str(Path(model_dir))
    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True)
    model.eval()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    return tok, model, device

def predict_multiclass(texts, tok, model, device, batch_size=16):
    id2label = getattr(model.config, 'id2label', None) or {}
    preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tok(batch, padding=True, truncation=True, max_length=512, return_tensors='pt')
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits
            ids = torch.argmax(logits, dim=-1).tolist()
            for idx in ids:
                raw = id2label.get(str(idx)) or id2label.get(idx) or str(idx)
                preds.append(norm_task1_label(raw))
    return preds

def resolve_pos_id(model):
    label2id = getattr(model.config, 'label2id', None) or {}
    for key in ['POS', 'CONTRIBUTION', 'YES', 'TRUE', '1']:
        if key in label2id:
            return int(label2id[key])
    return 1

def predict_binary(texts, tok, model, device, batch_size=16, threshold=0.5):
    pos_id = resolve_pos_id(model)
    preds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tok(batch, padding=True, truncation=True, max_length=512, return_tensors='pt')
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)
            if probs.shape[-1] == 1:
                p_pos = probs.squeeze(-1).tolist()
            else:
                p_pos = probs[:, pos_id].tolist()
            preds.extend([1 if float(p) >= threshold else 0 for p in p_pos])
    return preds

def plot_confusion(cm, labels, title):
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')
    ax.set_title(title)
    plt.xticks(rotation=35, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

## Task1: Matriz de confusión por clase

In [ ]:
df1 = pd.read_csv(TASK1_GOLD_CSV)
texts1 = [clean_text(t) for t in df1['text'].tolist()]
y1 = [norm_task1_label(y) for y in df1['gold_label'].tolist()]

# filtra filas válidas
pairs1 = [(t, y) for t, y in zip(texts1, y1) if t and y in TASK1_LABEL_ORDER]
texts1 = [p[0] for p in pairs1]
y1 = [p[1] for p in pairs1]

tok1, model1, dev1 = load_model(TASK1_MODEL_DIR)
p1 = predict_multiclass(texts1, tok1, model1, dev1, batch_size=TASK1_BATCH_SIZE)

# si sale label fuera de catálogo, lo mapeará a BACK para no romper matriz
p1 = [x if x in TASK1_LABEL_ORDER else 'BACK' for x in p1]

cm1 = confusion_matrix(y1, p1, labels=TASK1_LABEL_ORDER)
acc1 = accuracy_score(y1, p1)
rec1_macro = recall_score(y1, p1, labels=TASK1_LABEL_ORDER, average='macro', zero_division=0)

print(f'Task1 samples: {len(y1)}')
print(f'Task1 accuracy: {acc1:.4f}')
print(f'Task1 recall_macro: {rec1_macro:.4f}')
print(classification_report(y1, p1, labels=TASK1_LABEL_ORDER, zero_division=0))

plot_confusion(cm1, TASK1_LABEL_ORDER, 'Task1 - Confusion Matrix')

## Task2: Matriz de confusión (binaria)

In [ ]:
df2 = pd.read_csv(TASK2_GOLD_CSV)
texts2 = [clean_text(t) for t in df2['text'].tolist()]

# gold binario esperado en columna gold_is_contribution
def to_bin(x):
    s = str(x).strip().lower()
    if s in ['1', 'true', 't', 'yes', 'y', 'si', 'sí']:
        return 1
    if s in ['0', 'false', 'f', 'no', 'n']:
        return 0
    try:
        return 1 if int(float(s)) == 1 else 0
    except Exception:
        return None

y2 = [to_bin(x) for x in df2['gold_is_contribution'].tolist()]
pairs2 = [(t, y) for t, y in zip(texts2, y2) if t and y is not None]
texts2 = [p[0] for p in pairs2]
y2 = [p[1] for p in pairs2]

tok2, model2, dev2 = load_model(TASK2_MODEL_DIR)
p2 = predict_binary(texts2, tok2, model2, dev2, batch_size=TASK2_BATCH_SIZE, threshold=TASK2_THRESHOLD)

labels2 = [0, 1]
cm2 = confusion_matrix(y2, p2, labels=labels2)
acc2 = accuracy_score(y2, p2)
rec2 = recall_score(y2, p2, average='binary', pos_label=1, zero_division=0)

print(f'Task2 samples: {len(y2)}')
print(f'Task2 accuracy: {acc2:.4f}')
print(f'Task2 recall_pos: {rec2:.4f}')
print(classification_report(y2, p2, labels=labels2, target_names=['no_contribution', 'contribution'], zero_division=0))

plot_confusion(cm2, ['no_contribution', 'contribution'], 'Task2 - Confusion Matrix')

In [ ]:
# (Opcional) guardar matrices
out_dir = Path('/content/confusion_outputs')
out_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(cm1, index=TASK1_LABEL_ORDER, columns=TASK1_LABEL_ORDER).to_csv(out_dir / 'task1_confusion_matrix.csv')
pd.DataFrame(cm2, index=['no_contribution', 'contribution'], columns=['no_contribution', 'contribution']).to_csv(out_dir / 'task2_confusion_matrix.csv')
print('Saved in', out_dir)